
# 📦 Random Sampling from ModelNet40 Dataset

This script performs random sampling of `.off` files from the [ModelNet40](https://modelnet.cs.princeton.edu/) dataset. It uniformly samples a total of 1000 3D object files across all object categories, and saves them into a new folder structure while preserving class labels.

## 📁 Input and Output

- **Input**: Full ModelNet40 folder with subfolders structured as:
```

ModelNet40/
└── chair/
├── train/
│   ├── chair\_0001.off
│   └── ...
└── test/
├── chair\_1001.off
└── ...
└── table/
└── ...

````
- **Output**: A new folder `ModelNet40_sampled/` with uniformly sampled `.off` files distributed across the original categories.

## ⚙️ Configuration

```python
num_samples = 1000  # Total number of files to sample
samples_per_class = num_samples // len(categories)  # Equal sampling from each class
````

## 🚀 Key Functions

* 🔍 Searches both `train/` and `test/` folders for `.off` files.
* 🎲 Uses `random.sample()` for uniform sampling.
* 📂 Preserves category structure while copying files to a new output directory.

## ✅ Example Output

```
ModelNet40_sampled/
  └── chair/
      ├── chair_0003.off
      └── ...
  └── table/
      └── ...
```

## 📝 Notes

* If a class contains fewer files than `samples_per_class`, it selects all available files.
* Modify `data_dir` and `output_dir` to match your local environment.

## 📜 Full Script

See [`random_sample_modelnet40.py`](./random_sample_modelnet40.py) for the complete implementation.

```




In [ ]:
import os
import random
import shutil

# 🔹 Path to the ModelNet40 dataset (set to the folder you downloaded)
data_dir = "/Users/junseok/Downloads/ModelNet40"  # <<<<< Modify this to your ModelNet40 folder path!
output_dir = "/Users/junseok/Downloads/ModelNet40_sampled"  # <<<<< Folder to save the sampled data
os.makedirs(output_dir, exist_ok=True)

# 🔹 Get list of category folders
categories = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
print(f"Total {len(categories)} categories found.")

# 🔹 Configuration for selecting a total of 1000 samples
num_samples = 1000
samples_per_class = num_samples // len(categories)  # Select equal number of samples per category
selected_files = []

# 🔹 Sampling files per category
for category in categories:
    category_path = os.path.join(data_dir, category)

    # 🔹 Search for .off files under train/ or test/ subfolders
    off_files = []
    for sub_folder in ["train", "test"]:
        sub_path = os.path.join(category_path, sub_folder)
        if os.path.exists(sub_path):  # Only search if the train/ or test/ folder exists
            off_files.extend([
                os.path.join(sub_path, f)
                for f in os.listdir(sub_path)
                if f.lower().endswith(".off")
            ])

    # Determine how many files to sample from this category
    num_to_sample = min(samples_per_class, len(off_files))

    # Randomly sample files
    sampled_files = random.sample(off_files, num_to_sample)

    # Add to the overall list
    selected_files.extend(sampled_files)

print(f"Total number of sampled files: {len(selected_files)}")

# 🔹 Copy the selected samples to the new output directory
for src_path in selected_files:
    # Extract category name from the source path
    category = os.path.basename(os.path.dirname(os.path.dirname(src_path)))  # Get parent of train/test folder
    dst_category_path = os.path.join(output_dir, category)
    os.makedirs(dst_category_path, exist_ok=True)

    dst_path = os.path.join(dst_category_path, os.path.basename(src_path))
    shutil.copy(src_path, dst_path)

print(f"✅ Random sampling completed! Sampled data saved at: {output_dir}")


총 40개의 클래스가 발견됨.
총 선택된 파일 개수: 1000개
✅ 랜덤 샘플링 완료! 샘플 데이터가 /Users/junseok/Downloads/ModelNet40_sampled에 저장됨.




---

```markdown
# 🧪 3D Point Cloud Data Augmentation using Open3D

This script performs **data augmentation** on 3D point cloud files in `.off` format using the [Open3D](http://www.open3d.org/) library. The goal is to increase dataset diversity and robustness for training 3D deep learning models (e.g., for classification, detection, reconstruction).

## 🔧 Augmentation Techniques

The following four augmentations are applied sequentially to each point cloud:

| Augmentation | Description |
|--------------|-------------|
| **Rotation** | Random rotation around the Y-axis within ±15° |
| **Scaling**  | Uniform scaling within 0.9x to 1.1x |
| **Noise Injection** | Adds Gaussian noise (σ = 0.02) to each point |
| **Point Dropout** | Randomly drops 5–10% of points |

---

## 📁 Input/Output

- **Input Folder**: `/Users/junseok/Downloads/ModelNet40_sampled`
  - Structure: Each subfolder is a class, containing `.off` files
- **Output Folder**: `/Users/junseok/Downloads/ModelNet40_augmented`
  - Structure: Same as input, but each file is saved as `.ply` with `_augmented` suffix

Example:

```

ModelNet40\_sampled/
└── chair/
└── chair\_001.off

ModelNet40\_augmented/
└── chair/
└── chair\_001\_augmented.ply

````

---

## 🧠 Core Functions

```python
apply_rotation(pc)     # Rotate around Y-axis
apply_scaling(pc)      # Random scaling
apply_noise(pc)        # Add Gaussian noise
apply_dropout(pc)      # Drop some points
load_off_file(path)    # Load .off to Open3D point cloud
apply_augmentations(input_dir, output_dir)  # Apply and save
````

---

## 🚀 How to Use

1. Place your `.off` files into `ModelNet40_sampled/` structured by class.
2. Run the script with Open3D and NumPy installed.
3. Find the augmented `.ply` files in `ModelNet40_augmented/`.

### 📦 Requirements

```bash
pip install open3d numpy
```

---

## ✅ Output Message

After execution, you should see:

```
✅ All augmentations completed! Augmented data saved to: /path/to/ModelNet40_augmented
```

---

## 📌 Notes

* This script supports `.off` files with standard ModelNet40 formatting.
* You can modify `angle_range`, `scale_range`, `std_dev`, and `drop_ratio` in the function definitions to customize the augmentations.
* `.ply` is used as the output format for compatibility with most 3D learning frameworks.

```

---



In [ ]:
import open3d as o3d
import numpy as np
import os
import random

# 🔹 Rotation (±15 degrees around Y-axis)
def apply_rotation(point_cloud, angle_range=15):
    """Rotate the point cloud around the Y-axis by a random angle within ±angle_range degrees"""
    theta = np.radians(random.uniform(-angle_range, angle_range))  # Random angle in radians
    rot_matrix = np.array([[np.cos(theta), 0, np.sin(theta)],
                           [0, 1, 0],
                           [-np.sin(theta), 0, np.cos(theta)]])  # Rotation matrix around Y-axis
    point_cloud.rotate(rot_matrix, center=point_cloud.get_center())
    return point_cloud

# 🔹 Scaling (±10%)
def apply_scaling(point_cloud, scale_range=0.1):
    """Scale the point cloud randomly within ±scale_range (e.g., 0.9x to 1.1x)"""
    scale_factor = 1 + random.uniform(-scale_range, scale_range)
    point_cloud.scale(scale_factor, center=point_cloud.get_center())
    return point_cloud

# 🔹 Noise Injection (Gaussian noise with std dev <= 0.02)
def apply_noise(point_cloud, std_dev=0.02):
    """Add Gaussian noise to the point cloud"""
    points = np.asarray(point_cloud.points)
    noise = np.random.normal(0, std_dev, size=points.shape)
    points += noise
    point_cloud.points = o3d.utility.Vector3dVector(points)
    return point_cloud

# 🔹 Point Dropout (5–10%)
def apply_dropout(point_cloud, drop_ratio=0.1):
    """Randomly drop a portion of points from the point cloud"""
    points = np.asarray(point_cloud.points)
    num_points = len(points)
    num_drop = int(num_points * drop_ratio)

    # Randomly select indices to keep
    indices = np.random.choice(num_points, num_points - num_drop, replace=False)
    point_cloud.points = o3d.utility.Vector3dVector(points[indices])
    return point_cloud

def load_off_file(file_path):
    """Load a .off file and convert it to an Open3D PointCloud object"""
    with open(file_path, 'r') as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]  # Remove empty lines

    if not lines:
        raise ValueError(f"🚨 Error: File is empty - {file_path}")

    # Check and remove "OFF" header
    if lines[0].startswith("OFF"):
        lines[0] = lines[0].replace("OFF", "").strip()
        if lines[0] == "":
            lines.pop(0)

    if not lines:
        raise ValueError(f"🚨 Error: Could not read header from {file_path}")

    header = lines[0].split()

    try:
        num_vertices = int(header[0])
    except (ValueError, IndexError):
        raise ValueError(f"🚨 Invalid .off format in file: {file_path}, header={header}")

    # Read vertex coordinates
    vertices = []
    for i in range(1, num_vertices + 1):
        try:
            vertices.append(list(map(float, lines[i].split())))
        except ValueError:
            raise ValueError(f"🚨 Failed to parse line {i} in file {file_path}: {lines[i]}")

    point_cloud = o3d.geometry.PointCloud()
    point_cloud.points = o3d.utility.Vector3dVector(np.array(vertices))
    return point_cloud

# 🔹 Apply augmentations and save
def apply_augmentations(input_folder, output_folder):
    """Apply augmentations to all .off files in the folder and save them"""
    os.makedirs(output_folder, exist_ok=True)

    for category in os.listdir(input_folder):
        category_path = os.path.join(input_folder, category)
        if not os.path.isdir(category_path):
            continue

        output_category_path = os.path.join(output_folder, category)
        os.makedirs(output_category_path, exist_ok=True)

        for file in os.listdir(category_path):
            if file.endswith(".off"):
                file_path = os.path.join(category_path, file)
                point_cloud = load_off_file(file_path)

                # Apply all four augmentations
                point_cloud = apply_rotation(point_cloud)
                point_cloud = apply_scaling(point_cloud)
                point_cloud = apply_noise(point_cloud)
                point_cloud = apply_dropout(point_cloud)

                # Set output path and save as .ply
                output_path = os.path.join(output_category_path, file.replace(".off", "_augmented.ply"))
                o3d.io.write_point_cloud(output_path, point_cloud)

    print(f"✅ All augmentations completed! Augmented data saved to: {output_folder}")

# 🔹 Run (example paths)
input_folder = "/Users/junseok/Downloads/ModelNet40_sampled"
output_folder = "/Users/junseok/Downloads/ModelNet40_augmented"

apply_augmentations(input_folder, output_folder)



/data/jun/miniconda3/envs/Deeplearning_Assignment1/bin/python
